# PDF scan (OpenAI Vector Store + EVIDENCE-Pass)

Dieses Notebook:
1) lädt ein PDF in einen OpenAI Vector Store (hosted)
2) holt EVIDENCE-Auszüge via `vector_stores.search` und extrahiert daraus passende Stellen für dein Kapitel
3) gibt Treffer als **Suchanker** (für Strg+F) + Kurz-Zusammenfassung + Score aus (ohne Seitenzahlen)
4) zeigt am Ende eine Kostenübersicht (Token-basiert) für `gpt-5-nano`, `gpt-5-mini`, `gpt-5.2`

Voraussetzung: `OPENAI_API_KEY` ist in der Repo-`.env` oder als Umgebungsvariable gesetzt.


In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, Optional

from dotenv import load_dotenv
from openai import OpenAI

# .env laden (Repo root)
load_dotenv('.env', override=True)

api_key = (os.getenv('OPENAI_API_KEY') or '').strip()
if not api_key:
    raise RuntimeError('OPENAI_API_KEY fehlt. Setze ihn in der .env oder als Umgebungsvariable.')

client = OpenAI(api_key=api_key)

# --- Config (PDF Inputs) ---
# Option A: Explizite Liste (pfad ODER file_id)
# Beispiel:
# PDF_SOURCES = [
#     {'label': 'NIST ZTA', 'path': r'C:\\pdfs\\nist-zta.pdf'},
#     {'label': 'Paper XYZ', 'file_id': 'file-...'},
# ]
VECTOR_STORE_ID = (os.getenv('OPENAI_VECTOR_STORE_ID', '').strip() or '')

VECTOR_STORE_ID = 'vs_698213cec4508191ac5bbf2fcf97a3d7'
PDF_SOURCES = [
    {'label': 'test', 'file_id': 'file-CYXxpgpihxPo7muHs2nEVa'},
    {'label': 'test2', 'file_id': 'file-GLBt143zPp2hjzy5y7cLsh'},
    {'label': 'test3', 'file_id': 'file-DXezX5RKszhBrW2V6MqxLS'},
    {'label': 'test4', 'file_id': 'file-EL8cb4frcqZgxY6hgHKiM7'},
]

# Option B: Alle PDFs aus einem Ordner (wenn PDF_SOURCES leer ist)
PDF_DIR = r"static"  # TODO, z.B. r"C:\\pdfs"
PDF_GLOB = "*.pdf"
PDF_RECURSIVE = False
MAX_PDFS = 5  # Sicherheitslimit

# Option C: Single-PDF (wenn PDF_SOURCES leer ist und PDF_DIR leer ist)
PDF_PATH = r""  # TODO, z.B. r"C:\\pdfs\\paper.pdf"

# Optional: existierenden Vector Store / Files wiederverwenden (spart Upload + Indexing)
# Du kannst die IDs auch über Env-Variablen setzen.


# Backward-Compat: einzelnes OpenAI file_id
PDF_FILE_ID = (os.getenv('OPENAI_PDF_FILE_ID', '').strip() or '')

# Optional: mehrere file_ids via Env (CSV oder label=file_id)
# Beispiele:
#   OPENAI_PDF_FILE_IDS="file-abc,file-def"
#   OPENAI_PDF_FILE_IDS="NIST=file-abc,PaperXYZ=file-def"
PDF_FILE_IDS_RAW = (os.getenv('OPENAI_PDF_FILE_IDS', '').strip() or '')

# Modelle
MODEL = (os.getenv('OPENAI_PDF_SCAN_MODEL', 'gpt-5-mini') or '').strip() or 'gpt-5-mini'
PREPROCESS_MODEL = (os.getenv('OPENAI_PDF_SCAN_PREPROCESS_MODEL', 'gpt-5-nano') or '').strip() or 'gpt-5-nano'
PREPROCESS_FALLBACK_MODEL = (os.getenv('OPENAI_PDF_SCAN_PREPROCESS_FALLBACK_MODEL', 'gpt-5-mini') or '').strip() or 'gpt-5-mini'

# Modelle pro Stage (einfach ändern)
# Beispiele:
#   STAGE_MODELS['preprocess'] = 'gpt-5-nano'
#   STAGE_MODELS['preprocess_fallback'] = 'gpt-5-mini'
#   STAGE_MODELS['pdf_search'] = 'gpt-5.2'
STAGE_MODELS = {
    'preprocess': PREPROCESS_MODEL,
    'preprocess_fallback': PREPROCESS_FALLBACK_MODEL,
    'pdf_search': MODEL,
}


def model_for_stage(stage: str, default: Optional[str] = None) -> str:
    # 1) Notebook override via STAGE_MODELS
    v = (STAGE_MODELS.get(stage) or '').strip()
    if v:
        return v

    # 2) Optional env override, e.g. OPENAI_MODEL_PREPROCESS / OPENAI_MODEL_PDF_SEARCH
    env_key = f"OPENAI_MODEL_{stage.upper()}"
    v = (os.getenv(env_key) or '').strip()
    if v:
        return v

    # 3) Fallback
    return (default or MODEL or '').strip() or 'gpt-5-mini'

# Optional: Kapitelbeschreibung per LLM komprimieren/strukturieren
ENABLE_LLM_PREPROCESS = True

# Wichtig für gpt-5*: wenn max_output_tokens gesetzt ist, kann das Modell sonst in reines "reasoning"
# laufen und keine Antwort ausgeben. Mit reasoning.effort='low' kommt zuverlässig Output.
REASONING_EFFORT = (os.getenv('OPENAI_REASONING_EFFORT', 'low') or 'low').strip()

# Retrieval/Output Tuning
FILE_SEARCH_MAX_RESULTS_PER_PDF = 10  # 1-50; total max_num_results is capped at 50
MAX_HITS = 8
MAX_OUTPUT_TOKENS = 2500

# Preprocess Output-Limit (hoch genug, damit JSON nicht abgeschnitten wird)
PREPROCESS_MAX_OUTPUT_TOKENS = int(os.getenv('OPENAI_PREPROCESS_MAX_OUTPUT_TOKENS', '6000') or '6000')
PREPROCESS_RETRY_MAX_OUTPUT_TOKENS = int(os.getenv('OPENAI_PREPROCESS_RETRY_MAX_OUTPUT_TOKENS', '15000') or '15000')

# --- Pricing (USD / 1M tokens) ---
# Quelle: OpenAI Pricing (Stand: 2026-02-02)
MODEL_PRICING_USD_PER_1M = {
    'gpt-5-nano': {'input': 0.05, 'cached_input': 0.005, 'output': 0.40},
    'gpt-5-mini': {'input': 0.25, 'cached_input': 0.025, 'output': 2.00},
    'gpt-5.2': {'input': 1.75, 'cached_input': 0.175, 'output': 14.00},
}

COST_EVENTS = []  # wird über record_cost_event(...) befüllt


def _usage_int(obj: Any, key: str, default: int = 0) -> int:
    if obj is None:
        return default
    if isinstance(obj, dict):
        return int(obj.get(key, default) or 0)
    return int(getattr(obj, key, default) or 0)


def extract_usage(response: Any) -> Dict[str, int]:
    usage = getattr(response, 'usage', None)
    input_tokens = _usage_int(usage, 'input_tokens', 0)
    output_tokens = _usage_int(usage, 'output_tokens', 0)

    input_details = None
    if isinstance(usage, dict):
        input_details = usage.get('input_tokens_details')
    else:
        input_details = getattr(usage, 'input_tokens_details', None)

    cached_input_tokens = _usage_int(input_details, 'cached_tokens', 0)
    return {
        'input_tokens': int(input_tokens),
        'cached_input_tokens': int(cached_input_tokens),
        'output_tokens': int(output_tokens),
    }


def estimate_cost_usd(model: str, usage: Dict[str, int]) -> Optional[float]:
    prices = MODEL_PRICING_USD_PER_1M.get(model)
    if not prices:
        return None
    input_tokens = int(usage.get('input_tokens', 0) or 0)
    cached = int(usage.get('cached_input_tokens', 0) or 0)
    output_tokens = int(usage.get('output_tokens', 0) or 0)

    billable_input = max(0, input_tokens - cached)
    cost = (
        billable_input * float(prices['input'])
        + cached * float(prices['cached_input'])
        + output_tokens * float(prices['output'])
    ) / 1_000_000.0
    return float(cost)


def estimate_costs_for_all_priced_models(usage: Dict[str, int]) -> Dict[str, Optional[float]]:
    return {m: estimate_cost_usd(m, usage) for m in MODEL_PRICING_USD_PER_1M.keys()}


def record_cost_event(stage: str, response: Any) -> None:
    usage = extract_usage(response)
    COST_EVENTS.append(
        {
            'stage': stage,
            'model': getattr(response, 'model', None),
            'usage': usage,
            'costs_usd': estimate_costs_for_all_priced_models(usage),
        }
    )


def fmt_usd(x: Optional[float]) -> str:
    if x is None:
        return 'n/a'
    return f"${x:.6f}"


def get_response_text(response: Any) -> str:
    text = getattr(response, 'output_text', None)
    if isinstance(text, str) and text.strip():
        return text

    output = getattr(response, 'output', None)
    if output:
        for out_item in output:
            if isinstance(out_item, dict):
                content = out_item.get('content') or []
            else:
                content = getattr(out_item, 'content', None) or []
            for c in content:
                if isinstance(c, dict):
                    t = c.get('text')
                    if isinstance(t, str) and t.strip():
                        return t
                    if c.get('json') is not None:
                        return json.dumps(c.get('json'))
                    if c.get('parsed') is not None:
                        return json.dumps(c.get('parsed'))
                else:
                    t = getattr(c, 'text', None)
                    if isinstance(t, str) and t.strip():
                        return t
                    j = getattr(c, 'json', None)
                    if j is not None and not callable(j):
                        return json.dumps(j)
                    p = getattr(c, 'parsed', None)
                    if p is not None and not callable(p):
                        return json.dumps(p)

    return text if isinstance(text, str) else ''


def response_error_message(response: Any) -> Optional[str]:
    err = getattr(response, 'error', None)
    if not err:
        return None
    if isinstance(err, dict):
        msg = err.get('message')
        return (msg or str(err)).strip()
    msg = getattr(err, 'message', None)
    return (msg or str(err)).strip()


def poll_response_until_output(
    client: Any,
    response: Any,
    *,
    stage: str,
    timeout_s: int = 180,
    poll_interval_s: float = 1.0,
) -> Any:
    rid = getattr(response, 'id', None)
    if not rid:
        return response

    start = time.time()
    while True:
        if response_error_message(response):
            return response
        if (get_response_text(response) or '').strip():
            return response

        status = getattr(response, 'status', None)
        if status in {'completed', 'failed', 'cancelled', 'incomplete'}:
            return response

        if (time.time() - start) >= float(timeout_s):
            return response

        time.sleep(poll_interval_s)
        try:
            response = client.responses.retrieve(rid)
        except Exception:
            return response


def build_evidence_from_vector_store_search(
    search_page: Any,
    *,
    max_hits: int = 12,
    max_chars_per_hit: int = 1800,
) -> str:
    items = getattr(search_page, 'data', None)
    if items is None:
        try:
            items = list(search_page)
        except Exception:
            items = []

    evidence_parts = []
    for i, item in enumerate(list(items)[: max(0, int(max_hits))], start=1):
        score = getattr(item, 'score', None)
        if score is None:
            score = getattr(item, 'relevance_score', None)

        parts = []
        content = getattr(item, 'content', None)
        if content:
            for c in content:
                if isinstance(c, dict):
                    t = c.get('text')
                    if isinstance(t, str) and t.strip():
                        parts.append(t)
                else:
                    t = getattr(c, 'text', None)
                    if isinstance(t, str) and t.strip():
                        parts.append(t)

        text = '\n'.join(parts).strip()
        if not text:
            t = getattr(item, 'text', None)
            if isinstance(t, str) and t.strip():
                text = t.strip()

        text = ' '.join(text.split())
        if not text:
            continue

        if len(text) > int(max_chars_per_hit):
            text = text[: max(0, int(max_chars_per_hit))]
            if ' ' in text:
                text = text.rsplit(' ', 1)[0]
            text = text.strip()

        score_str = f"{float(score):.3f}" if score is not None else 'n/a'
        evidence_parts.append(f"[EVIDENCE {i} | score={score_str}]\n{text}")

    return '\n\n'.join(evidence_parts)


def parse_json_from_response(response: Any, stage: str) -> Dict[str, Any]:
    err_msg = response_error_message(response)
    if err_msg:
        rid = getattr(response, 'id', None)
        model = getattr(response, 'model', None)
        status = getattr(response, 'status', None)
        raise RuntimeError(
            f"{stage}: API error (id={rid}, model={model}, status={status}): {err_msg}"
        )

    # Prefer structured outputs if present (avoid returning callables like BaseModel.json)
    output = getattr(response, 'output', None)
    if output:
        for out_item in output:
            content = out_item.get('content') if isinstance(
                out_item, dict) else getattr(out_item, 'content', None)
            if not content:
                continue
            for c in content:
                if isinstance(c, dict):
                    parsed_val = c.get('parsed')
                    if isinstance(parsed_val, (dict, list)):
                        return parsed_val
                    json_val = c.get('json')
                    if isinstance(json_val, (dict, list)):
                        return json_val
                    if isinstance(parsed_val,
                                  str) and parsed_val.strip().startswith(
                                      ('{', '[')):
                        try:
                            return json.loads(parsed_val)
                        except Exception:
                            pass
                    if isinstance(json_val,
                                  str) and json_val.strip().startswith(
                                      ('{', '[')):
                        try:
                            return json.loads(json_val)
                        except Exception:
                            pass
                else:
                    parsed_val = getattr(c, 'parsed', None)
                    if callable(parsed_val):
                        parsed_val = None
                    if isinstance(parsed_val, (dict, list)):
                        return parsed_val
                    json_val = getattr(c, 'json', None)
                    if callable(json_val):
                        json_val = None
                    if isinstance(json_val, (dict, list)):
                        return json_val
                    if isinstance(parsed_val,
                                  str) and parsed_val.strip().startswith(
                                      ('{', '[')):
                        try:
                            return json.loads(parsed_val)
                        except Exception:
                            pass
                    if isinstance(json_val,
                                  str) and json_val.strip().startswith(
                                      ('{', '[')):
                        try:
                            return json.loads(json_val)
                        except Exception:
                            pass

    raw = (get_response_text(response) or '').strip()
    if not raw:
        rid = getattr(response, 'id', None)
        model = getattr(response, 'model', None)
        status = getattr(response, 'status', None)
        inc = getattr(response, 'incomplete_details', None)
        raise RuntimeError(
            f"{stage}: empty model output (id={rid}, model={model}, status={status}, incomplete_details={inc})."
        )

    # Strip common code-fence wrappers
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1] if '\n' in raw else ''
        raw = raw.rsplit('```', 1)[0] if '```' in raw else raw
        raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Robust fallback: sometimes models emit multiple JSON values (e.g. '{}{}').
        # We parse all JSON values we can find and pick the best candidate.
        decoder = json.JSONDecoder()

        starts = [0]
        for ch in ('{', '['):
            pos = raw.find(ch)
            if pos != -1:
                starts.append(pos)

        candidates = []
        for start in sorted(set(starts)):
            idx = int(start)
            local = []
            while idx < len(raw):
                while idx < len(raw) and raw[idx].isspace():
                    idx += 1
                if idx >= len(raw):
                    break
                try:
                    obj, end = decoder.raw_decode(raw, idx)
                except json.JSONDecodeError:
                    break
                local.append(obj)
                idx = int(end)
            if local:
                candidates = local
                break

        stage_l = (stage or '').lower()
        expected_keys = None
        if 'preprocess' in stage_l:
            expected_keys = [
                'optimized_description',
                'subpoints',
                'preferred_search_terms',
                'hard_exclusions',
                'must_terms',
                'should_terms',
                'scope_notes',
            ]
        elif 'search' in stage_l:
            expected_keys = [
                'none_found', 'primary_found', 'diagnostic', 'results'
            ]

        def has_expected_keys(obj: Any) -> bool:
            if not expected_keys:
                return False
            return isinstance(obj, dict) and all(k in obj
                                                 for k in expected_keys)

        for cand in reversed(candidates):
            if has_expected_keys(cand):
                return cand

        dict_candidates = [c for c in candidates if isinstance(c, dict)]
        if dict_candidates:
            merged = {}
            for c in dict_candidates:
                merged.update(c)
            if has_expected_keys(merged):
                return merged
            return dict_candidates[-1]

        raise


In [2]:
# --- Kapitel Input ---
# Du kannst diese beiden Variablen frei ändern.
# Hinweis: TITLE und DESCRIPTION werden später zu einem Such-Prompt kombiniert.
# In die DESCRIPTION gehört nur der inhaltliche Kapitel-Text (keine Such-/Output-Anweisungen).

USE_INTERACTIVE_INPUT = False

CHAPTER_TITLE = "Technische Grundlagen: Zero Trust Architecture (ZTA) in Unternehmensnetzwerken"

<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
"""

if USE_INTERACTIVE_INPUT:
    CHAPTER_TITLE = (input('Kapitel-Titel: ') or '').strip()
    print('Beschreibung eingeben (beenden mit leerer Zeile):')
    lines = []
    while True:
        line = input()
        if not line.strip():
            break
        lines.append(line)
    CHAPTER_DESCRIPTION = "\n".join(lines).strip()

if not CHAPTER_TITLE.strip():
    raise ValueError('CHAPTER_TITLE ist leer')
if not CHAPTER_DESCRIPTION.strip():
    raise ValueError('CHAPTER_DESCRIPTION ist leer')


In [3]:
# --- Preprocessing ---
# Ziel: lange Kapitelbeschreibung -> kompakte, gut durchsuchbare "Search Spec".

def normalize_whitespace(text: str) -> str:
    text = (text or '').replace('\r\n', '\n').replace('\r', '\n')
    lines = [ln.strip() for ln in text.split('\n')]
    lines = [ln for ln in lines if ln]
    return '\n'.join(lines).strip()


OPTIMIZED_DESCRIPTION = normalize_whitespace(CHAPTER_DESCRIPTION)
SUBPOINTS = []
PREFERRED_SEARCH_TERMS = []
MUST_TERMS = []
SHOULD_TERMS = []
SCOPE_NOTES = ''
HARD_EXCLUSIONS = []

if ENABLE_LLM_PREPROCESS:
    try:
        preprocess_schema = {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'optimized_description': {
                    'type': 'string',
                    'description': 'Kompakte such-optimierte Kapitelbeschreibung (Deutsch, max ~1200 Zeichen).',
                    'maxLength': 1200,
                },
                'must_terms': {
                    'type': 'array',
                    'description': 'Begriffe/Phrasen, die idealerweise im Treffer vorkommen (DE/EN gemischt erlaubt).',
                    'maxItems': 18,
                    'items': {'type': 'string', 'maxLength': 80},
                },
                'should_terms': {
                    'type': 'array',
                    'description': 'Unterstützende Begriffe/Synonyme/Keywords (DE/EN).',
                    'maxItems': 35,
                    'items': {'type': 'string', 'maxLength': 80},
                },
                'subpoints': {
                    'type': 'array',
                    'description': 'Unterpunkte wie (2.1) ... (falls vorhanden), sonst leere Liste.',
                    'maxItems': 25,
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'id': {'type': 'string', 'maxLength': 20},
                            'label': {'type': 'string', 'maxLength': 180},
                            'keywords': {
                                'type': 'array',
                                'description': 'Retrieval-Keywords je Subpoint (DE/EN erlaubt).',
                                'maxItems': 14,
                                'items': {'type': 'string', 'maxLength': 80},
                            },
                            'exclusions': {
                                'type': 'array',
                                'maxItems': 10,
                                'items': {'type': 'string', 'maxLength': 80},
                            },
                        },
                        'required': ['id', 'label', 'keywords', 'exclusions'],
                    },
                },
                'preferred_search_terms': {
                    'type': 'array',
                    'description': 'Kurze Liste von Keywords/Synonymen für Retrieval (DE/EN erlaubt).',
                    'maxItems': 30,
                    'items': {'type': 'string', 'maxLength': 80},
                },
                'hard_exclusions': {
                    'type': 'array',
                    'description': 'Was explizit NICHT rein soll (kurz).',
                    'maxItems': 30,
                    'items': {'type': 'string', 'maxLength': 100},
                },
                'scope_notes': {
                    'type': 'string',
                    'description': '1–3 kurze Sätze: woran man erkennt, dass ein Treffer wirklich im Scope ist.',
                    'maxLength': 280,
                },
            },
            'required': [
                'optimized_description',
                'subpoints',
                'preferred_search_terms',
                'hard_exclusions',
                'must_terms',
                'should_terms',
                'scope_notes',
            ],
        }

        system = (
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen>            'Regeln:\n'
            '- Schreibe auf Deutsch.\n'
            '- Gib NUR gültiges JSON zurück, exakt passend zum Schema.\n'
            '- optimized_description: 600–1200 Zeichen, nur Scope/Intent des Kapitels, keine Metakommentare.\n'
            '- Extrahiere Unterpunkte (z.B. 2.1, 2.2) NUR wenn sie explizit im Text stehen, sonst subpoints=[].\n'
            '- preferred_search_terms / must_terms / should_terms: DE/EN gemischt erlaubt (PDFs können gemischtsprachig sein).\n'
            '- Füge Synonyme/Keywords hinzu, aber erfinde keine neuen Themen außerhalb des Scopes.\n'
            '- hard_exclusions: kurze Negativbegriffe/Abschnitte (z.B. Referenzen/Anhang), wenn sie Retrieval stören.\n'
            '- scope_notes: 1–3 Sätze, woran man erkennt, dass ein Treffer wirklich im Scope ist.\n'
        )

        user = (
            f'Kapitel-Titel: {CHAPTER_TITLE}\n\n'
            f'Rohbeschreibung:\n{normalize_whitespace(CHAPTER_DESCRIPTION)}\n'
        )

        def run_preprocess_attempt(max_out: int, stage_name: str, model_name: str) -> Dict[str, Any]:
            resp = client.responses.create(
                background=False,
                model=model_name,
                reasoning={'effort': REASONING_EFFORT},
                input=[
                    {'role': 'system', 'content': [{'type': 'input_text', 'text': system}]},
                    {'role': 'user', 'content': [{'type': 'input_text', 'text': user}]},
                ],
                text={
                    'format': {
                        'type': 'json_schema',
                        'name': 'chapter_preprocess',
                        'schema': preprocess_schema,
                        'strict': True,
                    }
                },
                max_output_tokens=int(max_out),
            )
            resp = poll_response_until_output(client, resp, stage=stage_name)
            record_cost_event(stage_name, resp)
            parsed = parse_json_from_response(resp, stage_name)
            # If the API marks the response as incomplete due to max_output_tokens, but JSON is parseable and
            # matches our required shape, we still accept it (otherwise we'd unnecessarily warn+retry).
            return parsed

        pre_json = None
        preprocess_model = model_for_stage('preprocess', PREPROCESS_MODEL)
        try:
            pre_json = run_preprocess_attempt(PREPROCESS_MAX_OUTPUT_TOKENS, 'preprocess', preprocess_model)
        except Exception as e:
            print('Warning: LLM preprocessing failed; retrying once with higher max_output_tokens:', e)
            try:
                pre_json = run_preprocess_attempt(PREPROCESS_RETRY_MAX_OUTPUT_TOKENS, 'preprocess_retry', preprocess_model)
            except Exception as e2:
                fallback_model = model_for_stage('preprocess_fallback', PREPROCESS_FALLBACK_MODEL)
                if fallback_model and fallback_model != preprocess_model:
                    print(f"Warning: preprocess still failing; trying fallback model={fallback_model}:", e2)
                    pre_json = run_preprocess_attempt(PREPROCESS_RETRY_MAX_OUTPUT_TOKENS, 'preprocess_fallback', fallback_model)
                else:
                    raise

        OPTIMIZED_DESCRIPTION = (pre_json.get('optimized_description') or '').strip() or OPTIMIZED_DESCRIPTION
        SUBPOINTS = pre_json.get('subpoints') or []
        PREFERRED_SEARCH_TERMS = pre_json.get('preferred_search_terms') or []
        MUST_TERMS = pre_json.get('must_terms') or []
        SHOULD_TERMS = pre_json.get('should_terms') or []
        SCOPE_NOTES = (pre_json.get('scope_notes') or '').strip()
        HARD_EXCLUSIONS = pre_json.get('hard_exclusions') or []
    except Exception as e:
        print('Warning: LLM preprocessing failed; continuing without it:', e)

print('--- Optimized description ---')
print(OPTIMIZED_DESCRIPTION)
print('\n--- Subpoints ---')
for sp in SUBPOINTS:
    print(f"- {sp.get('id')}: {sp.get('label')}")
print('\n--- Preferred search terms ---')
print(', '.join(PREFERRED_SEARCH_TERMS[:30]))
print('\n--- Must terms ---')
print(', '.join(MUST_TERMS[:30]))
print('\n--- Should terms ---')
print(', '.join(SHOULD_TERMS[:30]))
print('\n--- Scope notes ---')
print(SCOPE_NOTES)
print('\n--- Hard exclusions ---')
print(', '.join(HARD_EXCLUSIONS[:30]))


--- Optimized description ---
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
--- Subpoints ---
- 1: Begriffsdefinition und Abgrenzung
- 2: Kernprinzipien
- 3: Referenzarchitekturen und Bausteine
- 4: Telemetrie und kontinuierliche Bewertung
- 5: Migration in Legacy-Umgebungen
- 6: Bewertungskriterien, Validierung, Grenzen

--- Preferred search terms ---
Zero Trust Architecture, ZTA reference architecture, policy decision point, microsegmentation enterprise, continuous authorization, identity-centric security, assume breach

--- Must terms ---
Zero Trust, Zero Trust Architecture, assume breach, least privilege, microsegmentation, Policy Decision Point, Policy Enforcement Point, Control Plane, Data Plane, continuous authorization

--- Should terms ---
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
--- Scope notes ---
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
--- Hard exclusions ---
Produktvergleich, Tool Buyers Guide, Penetration Testing How-To, Bloc

In [4]:
# --- Upload / Indexing (Multi-PDF, ein Vector Store) ---

def _parse_pdf_file_ids_env(raw: str) -> list:
    out = []
    for part in (raw or '').split(','):
        part = part.strip()
        if not part:
            continue
        if '=' in part:
            label, fid = part.split('=', 1)
            label = (label or '').strip()
            fid = (fid or '').strip()
        else:
            fid = part
            label = part
        if not fid:
            continue
        if not label:
            label = fid
        out.append({'label': label, 'file_id': fid})
    return out


def _discover_pdf_sources_from_dir() -> list:
    pdf_dir = (PDF_DIR or '').strip()
    if not pdf_dir:
        return []
    root = Path(pdf_dir).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"PDF_DIR not found: {root}")

    if bool(PDF_RECURSIVE):
        paths = sorted(root.rglob(PDF_GLOB))
    else:
        paths = sorted(root.glob(PDF_GLOB))
    paths = [p for p in paths if p.is_file()]
    if not paths:
        raise FileNotFoundError(f"No PDFs found in {root} with pattern {PDF_GLOB!r}")

    if len(paths) > int(MAX_PDFS):
        print(f"Note: Found {len(paths)} PDFs, using first {int(MAX_PDFS)}.")

    out = []
    for p in paths[: int(MAX_PDFS)]:
        out.append({'label': p.stem, 'path': str(p)})
    return out


def _normalize_pdf_sources(raw_sources: Any) -> list:
    out = []
    if not raw_sources:
        return out
    for s in raw_sources:
        if not isinstance(s, dict):
            continue
        label = (s.get('label') or '').strip()
        file_id = (s.get('file_id') or '').strip()
        path = (s.get('path') or '').strip()
        if not label:
            if path:
                label = Path(path).stem
            elif file_id:
                label = file_id
            else:
                label = 'pdf'
        out.append({'label': label, 'file_id': file_id, 'path': path})

    # Labels eindeutig machen
    seen = set()
    for s in out:
        base = s['label']
        label = base
        n = 2
        while label in seen:
            label = f"{base} ({n})"
            n += 1
        s['label'] = label
        seen.add(label)

    return out


# --- Quellen bestimmen (Single + Multi kompatibel) ---
pdf_sources = _normalize_pdf_sources(PDF_SOURCES)

if not pdf_sources and (PDF_FILE_IDS_RAW or '').strip():
    pdf_sources = _normalize_pdf_sources(_parse_pdf_file_ids_env(PDF_FILE_IDS_RAW))

if not pdf_sources and (PDF_FILE_ID or '').strip():
    pdf_sources = _normalize_pdf_sources([{'label': 'pdf', 'file_id': (PDF_FILE_ID or '').strip()}])

if not pdf_sources:
    pdf_sources = _normalize_pdf_sources(_discover_pdf_sources_from_dir())

if not pdf_sources and (PDF_PATH or '').strip():
    p = Path(PDF_PATH).expanduser().resolve()
    pdf_sources = _normalize_pdf_sources([{'label': p.stem or 'pdf', 'path': str(p)}])

if not pdf_sources:
    raise RuntimeError(
        "Keine PDFs konfiguriert. Setze entweder PDF_SOURCES, PDF_DIR oder PDF_PATH "
        "oder nutze OPENAI_PDF_FILE_ID / OPENAI_PDF_FILE_IDS."
    )

print('PDFs to process:')
for s in pdf_sources:
    print(f"  - {s.get('label')}: {s.get('path') or s.get('file_id')}")

# 1) Vector Store erstellen (falls nicht gesetzt)
if not (VECTOR_STORE_ID or '').strip():
    vector_store = client.vector_stores.create(
        name=f"pdf-scan:multi:{int(time.time())}",
        # Optional: automatisch ablaufen lassen, damit keine dauerhaften Storage-Kosten entstehen
        expires_after={"anchor": "last_active_at", "days": 30},
    )
    VECTOR_STORE_ID = vector_store.id
    print('Created vector store:', VECTOR_STORE_ID)
else:
    print('Using existing vector store:', VECTOR_STORE_ID)


def _ensure_attached_and_indexed(file_id: str) -> None:
    # 1) Try retrieve (works if already attached)
    vs_file = None
    try:
        vs_file = client.vector_stores.files.retrieve(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
    except Exception:
        vs_file = None

    # 2) Attach if missing
    if vs_file is None:
        if hasattr(client.vector_stores.files, 'create_and_poll'):
            try:
                vs_file = client.vector_stores.files.create_and_poll(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
            except Exception as e:
                print('Warning: attach failed, trying retrieve:', e)
                vs_file = client.vector_stores.files.retrieve(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
        else:
            try:
                client.vector_stores.files.create(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
            except Exception as e:
                print('Warning: attach failed (continuing):', e)
            vs_file = client.vector_stores.files.retrieve(vector_store_id=VECTOR_STORE_ID, file_id=file_id)

    # 3) Poll if still indexing
    status = getattr(vs_file, 'status', None)
    if status and status not in {'completed', 'failed'} and hasattr(client.vector_stores.files, 'poll'):
        vs_file = client.vector_stores.files.poll(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
        status = getattr(vs_file, 'status', status)

    if status == 'failed':
        raise RuntimeError(f"Vector store indexing failed for file_id={file_id}.")

    print('Vector store file status:', file_id, status)


# 2) Upload/Re-use + attach all PDFs
PDF_ARTIFACTS = []
for s in pdf_sources:
    label = s.get('label')
    file_id = (s.get('file_id') or '').strip()
    path_raw = (s.get('path') or '').strip()

    if not file_id:
        if not path_raw:
            raise RuntimeError(f"PDF source missing both path and file_id: {s}")
        path = Path(path_raw).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {path}")
        with path.open('rb') as f:
            file_obj = client.files.create(file=f, purpose='assistants')
        file_id = file_obj.id
        print(f"Uploaded '{label}' -> file_id={file_id}")
    else:
        print(f"Reusing '{label}' -> file_id={file_id}")

    _ensure_attached_and_indexed(str(file_id))
    PDF_ARTIFACTS.append({'label': str(label), 'file_id': str(file_id), 'path': path_raw})

PDF_LABEL_BY_FILE_ID = {a['file_id']: a['label'] for a in PDF_ARTIFACTS}

print('Reuse IDs:')
print('  VECTOR_STORE_ID =', VECTOR_STORE_ID)
print('  PDF_FILE_IDS    =', ', '.join([a['file_id'] for a in PDF_ARTIFACTS]))
print('Vector store ready:', VECTOR_STORE_ID)

print('\n--- Copy/paste for reuse (hardcoded) ---')
print(f"VECTOR_STORE_ID = '{VECTOR_STORE_ID}'")
print('PDF_SOURCES = [')
for a in PDF_ARTIFACTS:
    print(f"    {{'label': {a['label']!r}, 'file_id': {a['file_id']!r}}},")
print(']')

print('\n--- Or set env vars ---')
print(f"OPENAI_VECTOR_STORE_ID={VECTOR_STORE_ID}")
pairs = ','.join([f"{a['label']}={a['file_id']}" for a in PDF_ARTIFACTS])
print(f"OPENAI_PDF_FILE_IDS={pairs}")


PDFs to process:
  - test: file-CYXxpgpihxPo7muHs2nEVa
  - test2: file-GLBt143zPp2hjzy5y7cLsh
  - test3: file-DXezX5RKszhBrW2V6MqxLS
  - test4: file-EL8cb4frcqZgxY6hgHKiM7
Using existing vector store: vs_698213cec4508191ac5bbf2fcf97a3d7
Reusing 'test' -> file_id=file-CYXxpgpihxPo7muHs2nEVa
Vector store file status: file-CYXxpgpihxPo7muHs2nEVa completed
Reusing 'test2' -> file_id=file-GLBt143zPp2hjzy5y7cLsh
Vector store file status: file-GLBt143zPp2hjzy5y7cLsh completed
Reusing 'test3' -> file_id=file-DXezX5RKszhBrW2V6MqxLS
Vector store file status: file-DXezX5RKszhBrW2V6MqxLS completed
Reusing 'test4' -> file_id=file-EL8cb4frcqZgxY6hgHKiM7
Vector store file status: file-EL8cb4frcqZgxY6hgHKiM7 completed
Reuse IDs:
  VECTOR_STORE_ID = vs_698213cec4508191ac5bbf2fcf97a3d7
  PDF_FILE_IDS    = file-CYXxpgpihxPo7muHs2nEVa, file-GLBt143zPp2hjzy5y7cLsh, file-DXezX5RKszhBrW2V6MqxLS, file-EL8cb4frcqZgxY6hgHKiM7
Vector store ready: vs_698213cec4508191ac5bbf2fcf97a3d7

--- Copy/paste for reuse (har

In [5]:
# --- Query: Vector Store durchsuchen + strukturierte Ausgabe (ohne Seitenzahlen) ---

result_schema = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'none_found': {
            'type': 'boolean',
            'description': 'True nur wenn gar nichts Sinnvolles gefunden wurde (auch keine Fallback-6er).',
        },
        'primary_found': {
            'type': 'boolean',
            'description': 'True wenn mindestens ein Treffer mit score >= 7 vorhanden ist.',
        },
        'diagnostic': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'reason': {
                    'type': 'string',
                    'enum': ['no_evidence', 'no_scope_match', 'only_weak_evidence', 'mixed_or_unclear'],
                },
                'best_score': {'type': 'integer', 'minimum': 1, 'maximum': 10},
                'notes': {'type': 'string', 'maxLength': 280},
            },
            'required': ['reason', 'best_score', 'notes'],
        },
        'results': {
            'type': 'array',
            'maxItems': 30,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'subpoint': {'type': 'string'},
                    'score_1_to_10': {'type': 'integer', 'minimum': 1, 'maximum': 10},
                    'tier': {
                        'type': 'string',
                        'enum': ['primary', 'fallback'],
                        'description': 'primary = score>=7; fallback = score==6 (nur wenn keine primary existieren).',
                    },
                    'anchor': {'type': 'string', 'description': '8–20 Wörter, exakt aus EVIDENCE.'},
                    'anchor_alt': {
                        'type': 'string',
                        'description': 'Zweites wörtliches Zitat aus EVIDENCE (6–14 Wörter), als Backup.',
                    },
                    'locator_hint': {
                        'type': 'string',
                        'description': 'Kurzer Hinweis zum Wiederfinden (kein Zitat).',
                        'maxLength': 200,
                    },
                    'coverage': {
                        'type': 'string',
                        'description': 'Welche Teilaspekte des Kapitels/Subpoints der Treffer gut abdeckt (1 Satz).',
                        'maxLength': 200,
                    },
                    'summary': {'type': 'string', 'description': '2–4 Sätze, nur basierend auf EVIDENCE.'},
                    'score_rationale': {
                        'type': 'string',
                        'description': 'Sehr kurz: Warum diese Zahl (z.B. direct definition, detailed mechanism, only overview).',
                        'maxLength': 180,
                    },
                },
                'required': [
                    'subpoint',
                    'score_1_to_10',
                    'tier',
                    'anchor',
                    'anchor_alt',
                    'locator_hint',
                    'coverage',
                    'summary',
                    'score_rationale',
                ],
            },
        },
    },
    'required': ['none_found', 'primary_found', 'diagnostic', 'results'],
}

subpoints_block = ''
if SUBPOINTS:
    lines = []
    for sp in SUBPOINTS:
        sid = sp.get('id')
        label = sp.get('label')
        kws = sp.get('keywords') or []
        excl = sp.get('exclusions') or []
        kw_s = ', '.join([str(k) for k in kws[:14] if str(k).strip()])
        excl_s = ', '.join([str(x) for x in excl[:10] if str(x).strip()])
        line = f"- ({sid}) {label}".strip()
        if kw_s:
            line += f" | keywords: {kw_s}"
        if excl_s:
            line += f" | exclusions: {excl_s}"
        lines.append(line)
    subpoints_block = '\n'.join(lines)
else:
    subpoints_block = '- (Allgemein) Keine Unterpunkte erkannt (du kannst sie in der Beschreibung hinzufügen).'

terms_block = ''
if PREFERRED_SEARCH_TERMS:
    terms_block = ', '.join(PREFERRED_SEARCH_TERMS)

must_terms_block = ''
if MUST_TERMS:
    must_terms_block = ', '.join(MUST_TERMS)

should_terms_block = ''
if SHOULD_TERMS:
    should_terms_block = ', '.join(SHOULD_TERMS)

scope_notes_block = (SCOPE_NOTES or '').strip()

excl_block = ''
if HARD_EXCLUSIONS:
    excl_block = '\n'.join([f"- {x}" for x in HARD_EXCLUSIONS])

search_query = normalize_whitespace(
    f"{CHAPTER_TITLE}\n\n{OPTIMIZED_DESCRIPTION}\n\n"
    f"MUST: {must_terms_block}\n"
    f"SHOULD: {should_terms_block}\n"
    f"Preferred: {terms_block}\n\n"
    f"Scope notes: {scope_notes_block}\n\n"
    f"Subpoints:\n{subpoints_block}\n\n"
    f"Exclusions:\n{excl_block}"
)

# --- Global vector store search (one call across all PDFs) ---
search_items = []
per_pdf = max(1, min(50, int(FILE_SEARCH_MAX_RESULTS_PER_PDF)))
try:
    total = int(per_pdf) * max(1, len(PDF_ARTIFACTS))
    if int(total) > 50:
        print(f"Note: max_num_results capped at 50 (requested {int(total)}). Lower FILE_SEARCH_MAX_RESULTS_PER_PDF if needed.")
        total = 50
    search_page = client.vector_stores.search(
        vector_store_id=VECTOR_STORE_ID,
        query=search_query,
        max_num_results=int(total),
        rewrite_query=True,
    )
    items = getattr(search_page, 'data', None)
    if items is None:
        try:
            items = list(search_page)
        except Exception:
            items = []
    search_items = list(items or [])
except Exception as e:
    print('Warning: evidence retrieval failed:', e)
    search_items = []

def _item_file_id(item: Any) -> Optional[str]:
    fid = getattr(item, 'file_id', None)
    if isinstance(fid, str) and fid.strip():
        return fid.strip()
    if isinstance(fid, dict):
        v = fid.get('id') or fid.get('file_id')
        if isinstance(v, str) and v.strip():
            return v.strip()
    if fid is not None and hasattr(fid, 'id'):
        v = getattr(fid, 'id', None)
        if isinstance(v, str) and v.strip():
            return v.strip()

    f = getattr(item, 'file', None)
    if isinstance(f, str) and f.strip():
        return f.strip()
    if isinstance(f, dict):
        v = f.get('id') or f.get('file_id')
        if isinstance(v, str) and v.strip():
            return v.strip()
    if f is not None and hasattr(f, 'id'):
        v = getattr(f, 'id', None)
        if isinstance(v, str) and v.strip():
            return v.strip()

    md = getattr(item, 'metadata', None)
    if isinstance(md, dict):
        v = md.get('file_id') or md.get('file')
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None

file_ids = [a['file_id'] for a in (PDF_ARTIFACTS or [])]
items_by_file = {fid: [] for fid in file_ids}

unknown_items = 0
for it in search_items:
    fid = _item_file_id(it)
    if fid and fid in items_by_file:
        items_by_file[fid].append(it)
    else:
        unknown_items += 1

if unknown_items:
    print(f"Note: {unknown_items} search hits could not be mapped to a known file_id (ignored).")

EVIDENCE_MAX_HITS_PER_PDF = min(12, int(per_pdf))
evidence_by_file = {}
for a in PDF_ARTIFACTS:
    fid = a['file_id']
    evidence_by_file[fid] = build_evidence_from_vector_store_search(
        items_by_file.get(fid, []),
        max_hits=int(EVIDENCE_MAX_HITS_PER_PDF),
        max_chars_per_hit=1800,
    )

# Backward-Compat: single-evidence variable (wird unten durch Multi-PDF Output ersetzt)
first_file_id = (PDF_ARTIFACTS[0]['file_id'] if (PDF_ARTIFACTS or []) else None)
evidence = evidence_by_file.get(first_file_id, '') if first_file_id else ''

# --- Stage 2 (Evidence Extractor) pro PDF ---
PDF_STAGE2_OUTPUTS = []

<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
Du bekommst:
(1) Kapitel-Spezifikation (Titel, Scope, Keywords, Ausschlüsse, Unterpunkte)
(2) EVIDENCE-Auszüge (wörtliche Textstücke aus dem PDF-Index)

HÄRTESTE REGEL:
- Nutze AUSSCHLIESSLICH Text aus den EVIDENCE-Auszügen.
- Keine externen Fakten, keine Ergänzungen, keine Annahmen.
- Wenn etwas nicht eindeutig in EVIDENCE steht: NICHT aufnehmen.

Ziel:
Finde maximal {MAX_HITS} Stellen, die das Kapitel wirklich tragen.
Wichtig: Ein Treffer kann auch nur einen Teilaspekt perfekt abdecken. Dann muss coverage das klar benennen.

Anker-Regeln:
- anchor MUSS ein wörtliches Zitat aus EVIDENCE sein (8-20 Wörter).
- anchor_alt MUSS ein zweites wörtliches Zitat aus EVIDENCE sein (6-14 Wörter).
- Exakt kopieren (keine Änderungen, keine Ellipsen, keine Umstellungen).
- Zeilenumbrüche im Zitat zu normalen Leerzeichen machen.
- Wähle möglichst eine Überschrift oder eine sehr charakteristische Satzphrase, die man per Strg+F findet.
- locator_hint ist KEIN Zitat, sondern ein kurzer Hinweis (z.B. vermutete Abschnittsüberschrift + 2-4 Keywords). Wenn möglich, nenne auch eine Abschnittsnummer (z.B. 2.2).

Strenger Filter:
- Nur Treffer ausgeben, die klar im Scope liegen.
- Alles, was primär in den Ausschlüssen liegt: NICHT ausgeben.
- Wenn es Treffer mit score>=7 gibt: gib NUR solche als tier=primary aus.
- Wenn es KEINE Treffer mit score>=7 gibt: gib bis zu {MAX_HITS} Treffer mit score==6 als tier=fallback aus.
- Wenn nicht einmal 6 sinnvoll ist: none_found=true.

Output:
- Gib NUR JSON zurück, strikt passend zum Schema (keine Markdown-Fences, kein Zusatztext).
'''

for a in (PDF_ARTIFACTS or []):
    pdf_label = a.get('label')
    file_id = a.get('file_id')
    evidence = evidence_by_file.get(file_id, '')
    stage = f"pdf_search:{pdf_label}"

    if not evidence.strip():
        PDF_STAGE2_OUTPUTS.append(
            {
                'label': pdf_label,
                'file_id': file_id,
                'evidence': evidence,
                'data': {
                    'none_found': True,
                    'primary_found': False,
                    'diagnostic': {'reason': 'no_evidence', 'best_score': 1, 'notes': 'Keine EVIDENCE für dieses PDF.'},
                    'results': [],
                },
            }
        )
        continue

    USER2 = f'''### PDF
Label: {pdf_label}
OpenAI file_id: {file_id}

### Kapitel
Titel: {CHAPTER_TITLE}

### Such-Spezifikation (optimiert)
{OPTIMIZED_DESCRIPTION}

### Must-Terms (Core)
{must_terms_block}

### Should-Terms (Support)
{should_terms_block}

### Scope notes
{scope_notes_block}

### Unterpunkte (für Zuordnung)
{subpoints_block}

### Optionale Keywords/Synonyme
{terms_block}

### Ausschlüsse
{excl_block}

### Aufgabe
Analysiere die EVIDENCE-Auszüge und gib nur wirklich passende Stellen zurück.

### EVIDENCE
{evidence}
'''

    resp = client.responses.create(
        background=False,
        model=model_for_stage('pdf_search', MODEL),
        reasoning={'effort': REASONING_EFFORT},
        input=[
            {'role': 'system', 'content': [{'type': 'input_text', 'text': SYSTEM2}]},
            {'role': 'user', 'content': [{'type': 'input_text', 'text': USER2}]},
        ],
        text={
            'format': {
                'type': 'json_schema',
                'name': 'pdf_findings_evidence',
                'schema': result_schema,
                'strict': True,
            }
        },
        max_output_tokens=int(MAX_OUTPUT_TOKENS),
    )
    resp = poll_response_until_output(client, resp, stage=stage)
    record_cost_event(stage, resp)

    try:
        data = parse_json_from_response(resp, stage)
    except Exception as e:
        print('ERROR: could not parse JSON output from model:', e)
        print('--- Raw model output ---')
        print(get_response_text(resp))
        data = {
            'none_found': True,
            'primary_found': False,
            'diagnostic': {'reason': 'mixed_or_unclear', 'best_score': 1, 'notes': 'JSON parse error'},
            'results': [],
        }

    PDF_STAGE2_OUTPUTS.append({'label': pdf_label, 'file_id': file_id, 'evidence': evidence, 'data': data})

# --- Postprocess: Anchor validation + Thresholding + Combined outputs ---
import re
import unicodedata

def normalize_spaces(text: str) -> str:
    return ' '.join((text or '').split()).strip()

def norm_match(s: str) -> str:
    s = unicodedata.normalize('NFKC', s or '')
    s = s.replace('\u00ad', '')  # soft hyphen
    s = (
        s.replace('“', '"')
        .replace('”', '"')
        .replace('„', '"')
        .replace('‟', '"')
        .replace('’', "'")
        .replace('‘', "'")
        .replace('‛', "'")
        .replace('–', '-')
        .replace('—', '-')
        .replace('−', '-')
    )
    s = re.sub(r'\\s+', ' ', s).strip()
    return s

_STRIP_CHARS = "\"'“”„‟‘’‛()[]{}<>.,;:!?"

def norm_word(w: str) -> str:
    w = norm_match(w).lower()
    w = w.strip(_STRIP_CHARS)
    w = w.replace('-', '')
    return w

def _find_word_seq(hay: list, needle: list) -> Optional[int]:
    n = len(needle)
    if n <= 0:
        return None
    for i in range(0, len(hay) - n + 1):
        if hay[i : i + n] == needle:
            return int(i)
    return None

def _validate_anchor(anchor_raw: Any, evidence: str, *, min_words: int, max_words: int) -> Dict[str, Any]:
    raw = anchor_raw if isinstance(anchor_raw, str) else ''
    candidate = normalize_spaces(raw)
    if not candidate:
        return {'text': '', 'ok': False, 'reason': 'empty', 'span': None}
    if '…' in candidate or '...' in candidate:
        return {'text': candidate, 'ok': False, 'reason': 'ellipsis', 'span': None}
    words = candidate.split(' ')
    if len(words) < int(min_words) or len(words) > int(max_words):
        return {'text': candidate, 'ok': False, 'reason': 'word_count', 'span': None}

    evidence_norm = normalize_spaces(evidence)
    evidence_cmp = norm_match(evidence).lower()
    evidence_words = evidence_norm.split(' ') if evidence_norm else []
    evidence_words_cmp = [norm_word(w) for w in evidence_words]

    if evidence_words:
        needle = [norm_word(w) for w in words]
        start = _find_word_seq(evidence_words_cmp, needle)
        if start is not None:
            snapped = ' '.join(evidence_words[start : start + len(words)])
            return {'text': snapped, 'ok': True, 'reason': 'snapped', 'span': (int(start), int(len(words)))}

    cand_cmp = norm_match(candidate).lower()
    if evidence_cmp and cand_cmp and cand_cmp in evidence_cmp:
        return {'text': candidate, 'ok': True, 'reason': 'normalized_substring', 'span': None}

    return {'text': candidate, 'ok': False, 'reason': 'not_found_in_evidence', 'span': None}

def _derive_anchor_alt_from_span(evidence: str, span: Any, *, min_words: int = 6, max_words: int = 14) -> Optional[str]:
    if not span:
        return None
    words = normalize_spaces(evidence).split(' ')
    try:
        start, n = int(span[0]), int(span[1])
    except Exception:
        return None
    take = min(10, n, int(max_words))
    take = max(int(min_words), int(take))
    return ' '.join(words[start : start + take])

def _expand_span(evidence: str, span: Any, *, min_words: int = 8, max_words: int = 20) -> Optional[tuple]:
    if not span:
        return None
    words = normalize_spaces(evidence).split(' ')
    try:
        start, n = int(span[0]), int(span[1])
    except Exception:
        return None
    left = max(0, start)
    right = min(len(words), start + n)

    while (right - left) < int(min_words) and (right < len(words) or left > 0):
        if right < len(words):
            right += 1
        elif left > 0:
            left -= 1
        else:
            break

    if (right - left) > int(max_words):
        right = left + int(max_words)
    if (right - left) < int(min_words):
        return None
    return (int(left), int(right - left))

def postprocess_and_filter(data: Dict[str, Any], evidence: str) -> Dict[str, Any]:
    raw_results = data.get('results') or []
    validated_results = []
    dropped_invalid_shape = 0
    invalid_anchor = 0
    invalid_anchor_alt = 0
    derived_anchor = 0
    derived_anchor_alt = 0

    for r in raw_results:
        if not isinstance(r, dict):
            dropped_invalid_shape += 1
            continue

        try:
            score = int(r.get('score_1_to_10', 0) or 0)
        except Exception:
            score = 0

        a = _validate_anchor(r.get('anchor'), evidence, min_words=8, max_words=20)
        aa = _validate_anchor(r.get('anchor_alt'), evidence, min_words=6, max_words=14)

        if bool(a.get('ok')) and not bool(aa.get('ok')):
            derived = _derive_anchor_alt_from_span(evidence, a.get('span'))
            if derived:
                aa = {'text': derived, 'ok': True, 'reason': 'derived_from_anchor', 'span': None}
                derived_anchor_alt += 1

        if bool(aa.get('ok')) and not bool(a.get('ok')):
            exp = _expand_span(evidence, aa.get('span'))
            if exp:
                start, n = exp
                ev_words = normalize_spaces(evidence).split(' ')
                a = {'text': ' '.join(ev_words[start : start + n]), 'ok': True, 'reason': 'derived_from_anchor_alt', 'span': exp}
                derived_anchor += 1

        if not bool(a.get('ok')):
            invalid_anchor += 1
        if not bool(aa.get('ok')):
            invalid_anchor_alt += 1

        rr = dict(r)
        rr['score_1_to_10'] = int(score)
        rr['anchor'] = str(a.get('text') or '')
        rr['anchor_alt'] = str(aa.get('text') or '')
        rr['_anchor_ok'] = bool(a.get('ok'))
        rr['_anchor_reason'] = str(a.get('reason') or '')
        rr['_anchor_alt_ok'] = bool(aa.get('ok'))
        rr['_anchor_alt_reason'] = str(aa.get('reason') or '')
        validated_results.append(rr)

    prim = [r for r in validated_results if int(r.get('score_1_to_10', 0) or 0) >= 7]
    fallback = [r for r in validated_results if int(r.get('score_1_to_10', 0) or 0) == 6]

    if prim:
        keep = prim
        keep_tier = 'primary'
    else:
        keep = fallback
        keep_tier = 'fallback'

    for r in keep:
        r['tier'] = keep_tier

    keep = sorted(
        keep,
        key=lambda r: (
            int(r.get('score_1_to_10', 0) or 0),
            1 if (r.get('_anchor_ok') and r.get('_anchor_alt_ok')) else 0,
            1 if (r.get('_anchor_ok') or r.get('_anchor_alt_ok')) else 0,
        ),
        reverse=True,
    )[: int(MAX_HITS)]

    primary_found = bool(prim)
    none_found = not keep

    best_score = 1
    if keep:
        try:
            best_score = max(1, min(10, int(keep[0].get('score_1_to_10', 1) or 1)))
        except Exception:
            best_score = 1

    if not evidence.strip():
        reason = 'no_evidence'
    elif primary_found:
        reason = 'mixed_or_unclear'
    elif keep:
        reason = 'only_weak_evidence'
    else:
        reason = 'no_scope_match'

    notes = (
        f"raw={len(raw_results)}, kept={len(keep)}, primary={len(prim)}, fallback={len(fallback)}; "
        f"invalid_anchor={invalid_anchor}, invalid_anchor_alt={invalid_anchor_alt}; "
        f"derived_anchor={derived_anchor}, derived_anchor_alt={derived_anchor_alt}"
    )
    if dropped_invalid_shape:
        notes += f"; dropped_invalid_shape={dropped_invalid_shape}"

    keep_clean = [{k: v for k, v in r.items() if not str(k).startswith('_')} for r in keep]
    clean_data = dict(data)
    clean_data['results'] = keep_clean
    clean_data['primary_found'] = bool(primary_found)
    clean_data['none_found'] = bool(none_found)
    clean_data['diagnostic'] = {'reason': reason, 'best_score': int(best_score), 'notes': (notes or '')[:280]}

    stats = {
        'dropped_invalid_shape': dropped_invalid_shape,
        'invalid_anchor': invalid_anchor,
        'invalid_anchor_alt': invalid_anchor_alt,
        'derived_anchor': derived_anchor,
        'derived_anchor_alt': derived_anchor_alt,
    }
    return {'clean_data': clean_data, 'keep_debug': keep, 'stats': stats}


# --- Run postprocess per PDF ---
PDF_RESULTS = []
for item in (PDF_STAGE2_OUTPUTS or []):
    label = item.get('label')
    file_id = item.get('file_id')
    evidence = item.get('evidence') or ''
    data = item.get('data') or {}
    out = postprocess_and_filter(data, evidence)
    PDF_RESULTS.append(
        {
            'label': label,
            'file_id': file_id,
            'clean_data': out['clean_data'],
            'keep_debug': out['keep_debug'],
            'stats': out['stats'],
        }
    )


# --- Output: pro PDF ---
print('\n=== Ergebnisse pro PDF ===')
for pdf in PDF_RESULTS:
    label = pdf['label']
    file_id = pdf['file_id']
    data = pdf['clean_data']
    keep = pdf['keep_debug']
    stats = pdf['stats']

    print(f"\n--- PDF: {label} (file_id={file_id}) ---")
    if data.get('none_found'):
        print('KEINE PASSENDEN STELLEN GEFUNDEN')
        print('Diagnostic:', data.get('diagnostic'))
        continue

    if stats.get('dropped_invalid_shape') or stats.get('invalid_anchor') or stats.get('invalid_anchor_alt'):
        print(
            f"Note: dropped_invalid_shape={stats.get('dropped_invalid_shape')}, invalid_anchor={stats.get('invalid_anchor')}, "
            f"invalid_anchor_alt={stats.get('invalid_anchor_alt')}, derived_anchor={stats.get('derived_anchor')}, "
            f"derived_anchor_alt={stats.get('derived_anchor_alt')}"
        )

    for r in keep:
        print(f"- Unterpunkt: {r.get('subpoint')}")
        print(f"  - Tier: {r.get('tier')}")
        print(f"  - Score (1-10): {r.get('score_1_to_10')}")
        print(f"  - Coverage: {r.get('coverage')}")
        print(f"  - Ort (Anker): \"{r.get('anchor')}\"")
        print(f"  - Ort (Anker 2): \"{r.get('anchor_alt')}\"")
        if (not r.get('_anchor_ok')) or (not r.get('_anchor_alt_ok')):
            print(
                f"  - Anchor-Validierung: anchor_ok={r.get('_anchor_ok')} ({r.get('_anchor_reason')}), "
                f"anchor_alt_ok={r.get('_anchor_alt_ok')} ({r.get('_anchor_alt_reason')})"
            )
        print(f"  - Locator hint: {r.get('locator_hint')}")
        print(f"  - Score rationale: {r.get('score_rationale')}")
        print(f"  - Kurz-Zusammenfassung: {r.get('summary')}")
        print('')


# --- Output: nach Unterpunkt (über alle PDFs) ---
by_subpoint = {}
for pdf in PDF_RESULTS:
    for r in (pdf.get('keep_debug') or []):
        sp = (r.get('subpoint') or '(Allgemein)').strip() or '(Allgemein)'
        rr = dict(r)
        rr['pdf_label'] = pdf.get('label')
        rr['pdf_file_id'] = pdf.get('file_id')
        by_subpoint.setdefault(sp, []).append(rr)

print('\n=== Zusammenfassung nach Unterpunkt ===')
if not by_subpoint:
    print('(keine Treffer)')
else:
    for sp, hits in sorted(by_subpoint.items(), key=lambda kv: kv[0]):
        hits = sorted(hits, key=lambda r: int(r.get('score_1_to_10', 0) or 0), reverse=True)
        print(f"\n## {sp}")
        for r in hits[: int(MAX_HITS)]:
            unverified = '' if (r.get('_anchor_ok') and r.get('_anchor_alt_ok')) else ' [UNVERIFIED_ANCHOR]'
            print(f"- [{r.get('pdf_label')}] Score {r.get('score_1_to_10')} ({r.get('tier')}){unverified}")
            print(f"  Ort: \"{r.get('anchor')}\"")
            print(f"  Kurz: {r.get('summary')}\n")


# --- Combined output (für Weiterverarbeitung) ---
RUN_OUTPUT = {
    'vector_store_id': VECTOR_STORE_ID,
    'chapter_title': CHAPTER_TITLE,
    'pdfs': [
        {
            'label': pdf['label'],
            'file_id': pdf['file_id'],
            'data': pdf['clean_data'],
        }
        for pdf in PDF_RESULTS
    ],
    'by_subpoint': {
        sp: [{k: v for k, v in r.items() if not str(k).startswith('_')} for r in hits]
        for sp, hits in by_subpoint.items()
    },
}



=== Ergebnisse pro PDF ===

--- PDF: test (file_id=file-CYXxpgpihxPo7muHs2nEVa) ---
Note: dropped_invalid_shape=0, invalid_anchor=2, invalid_anchor_alt=2, derived_anchor=1, derived_anchor_alt=1
- Unterpunkt: (1) Begriffsdefinition und Abgrenzung
  - Tier: primary
  - Score (1-10): 9
  - Coverage: Defines ZTA and contrasts it with perimeter-based 'castle-and-moat' security models.
  - Ort (Anker): "Zero Trust Architecture (ZTA) represents a significant paradigm shift in cybersecurity,"
  - Ort (Anker 2): "assumes no implicit trust for any entity,"
  - Locator hint: Introduction; definition of ZTA, contrast to perimeter model (intro section)
  - Score rationale: Direct definitional statement and explicit contrast to perimeter approach in introduction.
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
- Unterpunkt: (2) Kernprinzipien
  - Tier: primary
  - Score (1-10): 9
  - Coverage: States core principles: continuous verification, least privilege, strict segmentation, and use o

In [6]:
# --- Cost summary ---
# Wichtig: Das sind Token-Kosten (LLM). Vector-Store Storage/Indexing-Kosten sind NICHT enthalten.

print('--- Pricing (USD / 1M tokens) ---')
for m, p in MODEL_PRICING_USD_PER_1M.items():
    print(f"- {m}: input=${p['input']}/1M, cached_input=${p['cached_input']}/1M, output=${p['output']}/1M")

def pricing_key_for_model_used(model_used: Optional[str]) -> Optional[str]:
    if not model_used:
        return None
    if model_used in MODEL_PRICING_USD_PER_1M:
        return model_used
    # Prefer longest prefix match (handles snapshot names like gpt-5-mini-YYYY-MM-DD)
    matches = [k for k in MODEL_PRICING_USD_PER_1M.keys() if model_used.startswith(k)]
    if not matches:
        return None
    return sorted(matches, key=len, reverse=True)[0]

actual_total = 0.0
actual_shared_preprocess = 0.0
actual_by_pdf = {}
actual_other = 0.0

print('\n--- Actual run costs (model_used pricing) ---')
for ev in COST_EVENTS:
    stage = str(ev.get('stage') or '')
    model_used = ev.get('model')
    usage = ev.get('usage') or {}
    costs = ev.get('costs_usd') or {}

    pricing_key = pricing_key_for_model_used(model_used)
    actual_cost = costs.get(pricing_key) if pricing_key else None
    if actual_cost is not None:
        actual_total += float(actual_cost)

        if stage.startswith('pdf_search:'):
            pdf_label = (stage.split(':', 1)[1] or '').strip() or '(unknown)'
            actual_by_pdf[pdf_label] = float(actual_by_pdf.get(pdf_label, 0.0)) + float(actual_cost)
        elif stage.startswith('preprocess'):
            actual_shared_preprocess += float(actual_cost)
        else:
            actual_other += float(actual_cost)

    print(f"Stage: {stage} | model_used={model_used} | priced_as={pricing_key or 'n/a'}")
    print(
        f"  tokens: input={usage.get('input_tokens', 0)} "
        f"(cached={usage.get('cached_input_tokens', 0)}), output={usage.get('output_tokens', 0)}"
    )
    print(f"  actual_cost: {fmt_usd(actual_cost)}")
    print('')

print(f"Total actual: ${actual_total:.6f}")

print('\n--- Actual breakdown (per PDF + total) ---')
if actual_shared_preprocess:
    print(f"- shared_preprocess: ${actual_shared_preprocess:.6f}")
if actual_other:
    print(f"- other_stages: ${actual_other:.6f}")
if actual_by_pdf:
    for pdf_label in sorted(actual_by_pdf.keys()):
        print(f"- pdf[{pdf_label}]: ${float(actual_by_pdf[pdf_label]):.6f}")
    print(f"- sum_pdfs: ${sum(float(v) for v in actual_by_pdf.values()):.6f}")
print(f"- total: ${actual_total:.6f}")

what_if_totals = {m: 0.0 for m in MODEL_PRICING_USD_PER_1M.keys()}
for ev in COST_EVENTS:
    costs = ev.get('costs_usd') or {}
    for m in MODEL_PRICING_USD_PER_1M.keys():
        c = costs.get(m)
        if c is not None:
            what_if_totals[m] += float(c)

print('\n--- What-if totals (same token counts, different pricing) ---')
for m, total in what_if_totals.items():
    print(f"- total@{m}: ${total:.6f}")


--- Pricing (USD / 1M tokens) ---
- gpt-5-nano: input=$0.05/1M, cached_input=$0.005/1M, output=$0.4/1M
- gpt-5-mini: input=$0.25/1M, cached_input=$0.025/1M, output=$2.0/1M
- gpt-5.2: input=$1.75/1M, cached_input=$0.175/1M, output=$14.0/1M

--- Actual run costs (model_used pricing) ---
Stage: preprocess | model_used=gpt-5-nano-2025-08-07 | priced_as=gpt-5-nano
  tokens: input=1103 (cached=0), output=1098
  actual_cost: $0.000494

Stage: preprocess_retry | model_used=gpt-5-nano-2025-08-07 | priced_as=gpt-5-nano
  tokens: input=1103 (cached=0), output=480
  actual_cost: $0.000247

Stage: preprocess_fallback | model_used=gpt-5-mini-2025-08-07 | priced_as=gpt-5-mini
  tokens: input=1103 (cached=0), output=1253
  actual_cost: $0.002782

Stage: pdf_search:test | model_used=gpt-5-mini-2025-08-07 | priced_as=gpt-5-mini
  tokens: input=4998 (cached=0), output=1536
  actual_cost: $0.004321

Stage: pdf_search:test2 | model_used=gpt-5-mini-2025-08-07 | priced_as=gpt-5-mini
  tokens: input=2670 (cac